# Urban AI Planning - Data Loading 
## Loading Partitioned Parquet Data from LinkUp Dataset

## Step 1: Import Libraries & Setup

In [5]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import warnings
import os
warnings.filterwarnings('ignore')

print(f"Current directory: {os.getcwd()}")

Current directory: C:\Users\Nikhil\OneDrive\Documents\urban-ai-planning\notebooks


## Step 2: Helper Function for Loading Data

This function handles:
- Loading partitioned parquet files
- Converting column names to lowercase (important!)
- Error handling

In [3]:
def load_parquet_with_lowercase(path, table_name="table"):
    """
    Load partitioned parquet files and convert columns to lowercase.
    
    Args:
        path: Path to folder containing parquet files
        table_name: Name for display purposes
    
    Returns:
        DataFrame with lowercase column names
    """
    try:
        print(f"Loading {table_name}...", end=" ")
        
        # Load all parquet files in folder
        df = pd.read_parquet(path)
        
        # Convert column names to lowercase
        df.columns = df.columns.str.lower()
        
        print(f"✓ Loaded {len(df):,} rows")
        return df
        
    except Exception as e:
        print(f"✗ Error: {e}")
        return None


## Step 3: Configure Paths

In [14]:
# Base path (up one level from notebooks folder)
BASE_PATH = Path('../data/raw')

# The three occupational categories
categories = [
    'extracted_transportation_engineers',
    'extracted_transportation_planners',
    'extracted_urban_regional_planning'
]

print("PATH VERIFICATION")
print("=" * 60)
print(f"Current directory: {Path.cwd()}")
print(f"Data path: {BASE_PATH.absolute()}")
print(f"Path exists: {BASE_PATH.exists()}")

if not BASE_PATH.exists():
    print("\nERROR: Data folder not found!")
else:
    print("\n Data folder found!")
    print("\nCategories to load:")
    for cat in categories:
        cat_path = BASE_PATH / cat
        exists = "✓" if cat_path.exists() else "✗"
        print(f"  {exists} {cat}")

PATH VERIFICATION
Current directory: C:\Users\Nikhil\OneDrive\Documents\urban-ai-planning\notebooks
Data path: C:\Users\Nikhil\OneDrive\Documents\urban-ai-planning\notebooks\..\data\raw
Path exists: True

 Data folder found!

Categories to load:
  ✓ extracted_transportation_engineers
  ✓ extracted_transportation_planners
  ✓ extracted_urban_regional_planning


## Step 4: Explore One Category First

Let's start by exploring Transportation Engineers to verify everything works.

In [16]:
# Test with first category
test_category = 'extracted_transportation_engineers'
test_path = BASE_PATH / test_category

print(f"Testing with: {test_category}")
print("=" * 60)

# Check what folders exist
folders = [f.name for f in test_path.iterdir() if f.is_dir()]
print(f"\nFolders found: {len(folders)}")

for folder in sorted(folders):
    folder_path = test_path / folder
    num_files = len(list(folder_path.glob('*.parquet')))
    print(f"  📁 {folder}")
    print(f"     Contains {num_files} parquet files")

Testing with: extracted_transportation_engineers

Folders found: 6
  📁 full-time-part-time
     Contains 32 parquet files
  📁 job-descriptions
     Contains 2160 parquet files
  📁 job-records
     Contains 251 parquet files
  📁 onet-taxonomy
     Contains 64 parquet files
  📁 remote-tag
     Contains 64 parquet files
  📁 structured-fields
     Contains 32 parquet files


## Step 5: Load Job Records

Load the core job records table with all metadata.

In [17]:
# Load job records (main table)
job_records = load_parquet_with_lowercase(
    test_path / 'job-records',
    table_name="job records"
)

if job_records is not None:
    print("\nJob Records Summary:")
    print("=" * 60)
    print(f"Total rows: {len(job_records):,}")
    print(f"Total columns: {len(job_records.columns)}")
    print(f"\nColumns: {list(job_records.columns)}")
    print(f"\nMemory usage: {job_records.memory_usage(deep=True).sum() / 1_000_000:.1f} MB")
    
    # Show sample
    display(job_records.head())

Loading job records... ✓ Loaded 36,379 rows

Job Records Summary:
Total rows: 36,379
Total columns: 17

Columns: ['base_hash', 'city', 'company_id', 'company_name', 'country', 'created', 'delete_date', 'job_hash', 'last_checked', 'last_updated', 'state', 'title', 'unmapped_location', 'url', 'zip', '_meta_loaded_at', 'onet_occupation_code']

Memory usage: 34.6 MB


,base_hash,city,company_id,company_name,country,created,delete_date,job_hash,last_checked,last_updated,state,title,unmapped_location,url,zip,_meta_loaded_at,onet_occupation_code
0,0aef31d7c9b65ffa2bf44b8b0aa9aeee,Denver,9694,Aecom Technology Corporation,USA,2012-07-27 14:20:00,2012-09-06 21:52:00,0aef31d7c9b65ffa2bf44b8b0aa9aeee,2012-09-05 13:37:00,NaT,CO,Traffic Design Engineer - Roadways,FALSE,https://jobs.aecom.com//1033/ASP/TG/cim_jobdet...,80208,2025-10-01 11:55:15.050,17-2051.01
1,0af1a44b3f4492b36d4e0431aa17754e,Austin,9694,Aecom Technology Corporation,USA,2013-12-24 23:43:00,2014-02-05 03:29:00,0af1a44b3f4492b36d4e0431aa17754e,2014-02-03 21:59:00,NaT,TX,Highway/Roadway/Transit EIT,FALSE,https://jobs.aecom.com/1033/ASP/TG/cim_jobdeta...,78719,2025-10-01 11:55:15.050,17-2051.01
2,0af450236a2b4f7b001a2fb57cfc08e1,None,22727,CenturyLink,USA,2015-11-12 04:01:00,2016-01-06 04:05:00,0af450236a2b4f7b001a2fb57cfc08e1,2016-01-05 04:03:00,NaT,None,Engineer III - Federal,FALSE,https://sjobs.brassring.com/TGWebHost/jobdetai...,None,2025-10-01 11:55:15.050,17-2051.01
3,0af756758d49beb883bdf46ad01874c8,New York City,12737,URS Corporation,USA,2009-11-06 03:18:00,2009-12-07 03:06:00,0af756758d49beb883bdf46ad01874c8,2009-12-07 03:06:00,NaT,NY,Transportation/Traffic Intern,FALSE,https://www.urs.apply2jobs.com/index.cfm?fusea...,10178,2025-10-01 11:55:15.050,17-2051.01
4,0af9de0777b1009e8a720dc6543da7ae,Anchorage,27197,State of Kentucky,USA,2015-02-13 17:18:00,2015-02-23 09:20:00,0af9de0777b1009e8a720dc6543da7ae,2015-02-20 11:04:00,NaT,AK,9046 Transportation Engineer Supervisor,FALSE,https://sjobs.brassring.com/TGWebHost/jobdetai...,99501,2025-10-01 11:55:15.050,17-2051.01


## Step 6: Load Job Descriptions

In [19]:
# Load job descriptions

job_descriptions = load_parquet_with_lowercase(
    test_path / 'job-descriptions',
    table_name="job descriptions"
)

if job_descriptions is not None:
    print("\nJob Descriptions Summary:")
    print("=" * 60)
    print(f"Total rows: {len(job_descriptions):,}")
    print(f"Columns: {list(job_descriptions.columns)}")
    
    # Check how many have actual descriptions
    with_text = job_descriptions['job_description'].notna().sum()
    print(f"\nJobs with descriptions: {with_text:,} ({with_text/len(job_descriptions)*100:.1f}%)")
    
    # Show average description length
    avg_length = job_descriptions['job_description'].str.len().mean()
    print(f"Average description length: {avg_length:.0f} characters")
    
    # Show sample
    display(job_descriptions.head())

Loading job descriptions... ✓ Loaded 24,111 rows

Job Descriptions Summary:
Total rows: 24,111
Columns: ['job_description', 'job_hash', 'onet_occupation_code']

Jobs with descriptions: 24,111 (100.0%)
Average description length: 4544 characters


,job_description,job_hash,onet_occupation_code
0,"About Us\n \nAt HDR, we specialize in enginee...",ab33012f1224050f2cf2ee064d095f70,17-2051.01
1,Job Description and Duties\n \nUnder the gene...,8817a322c43150697c9f4f7c9591e2bd,17-2051.01
2,POSITION SUMMARY:\n \nThe Project Engineer pr...,d0abff1a6331bf2d6a78c545d8a58883,17-2051.01
3,Overview\n\nHNTB is a firm where you can take ...,81e0d05482134b1206354de48d9448f5,17-2051.01
4,Just imagine your future with us\n \nAt Aurec...,60a2d9df8ba9b5482b70176a7e035011,17-2051.01


## Step 7: Join Job Records with Descriptions

In [20]:
if job_records is not None and job_descriptions is not None:
    # Join on job_hash
    combined = job_records.merge(
        job_descriptions,
        on='job_hash',
        how='inner',  # Only keep jobs with descriptions
        suffixes=('', '_desc')  # Handle duplicate columns
    )
    
    print("After Joining:")
    print("=" * 60)
    print(f"Job records: {len(job_records):,}")
    print(f"Job descriptions: {len(job_descriptions):,}")
    print(f"Combined (inner join): {len(combined):,}")
    print(f"\nTotal columns: {len(combined.columns)}")
    
    # Check for ONET code
    if 'onet_occupation_code' in combined.columns:
        print(f"\n✓ ONET codes present: {combined['onet_occupation_code'].notna().sum():,}")
        print(f"Top ONET codes:")
        print(combined['onet_occupation_code'].value_counts().head())
    
    display(combined.head())
else:
    print("Cannot join - one or both tables failed to load")

After Joining:
Job records: 36,379
Job descriptions: 24,111
Combined (inner join): 24,111

Total columns: 19

✓ ONET codes present: 24,111
Top ONET codes:
onet_occupation_code
17-2051.01    24111
Name: count, dtype: int64


,base_hash,city,company_id,company_name,country,created,delete_date,job_hash,last_checked,last_updated,state,title,unmapped_location,url,zip,_meta_loaded_at,onet_occupation_code,job_description,onet_occupation_code_desc
0,0afbae4c718319e9ca550c7d191257ce,Richland,32064,City of Richland,USA,2017-03-16 04:08:00,2017-06-21 21:28:00,0afbae4c718319e9ca550c7d191257ce,2017-06-19 20:40:00,NaT,WA,Traffic Engineer,FALSE,http://agency.governmentjobs.com/richlandwa/de...,99354,2025-10-01 11:55:15.050,17-2051.01,General Summary Benefits Supplemental Question...,17-2051.01
1,0afbfea6839cae2188306dc3de739c00,santa ana,975,State of California,USA,2023-07-13 08:34:00,2023-07-27 04:01:00,0afbfea6839cae2188306dc3de739c00,2023-07-24 10:51:00,NaT,CA,Transportation Engineer (civil),FALSE,https://www.calcareers.ca.gov/CalHrPublic/Jobs...,92701,2025-10-01 11:55:15.050,17-2051.01,Job Description and Duties\n \nUnder the gene...,17-2051.01
2,0afc1c76dee6174b12ec4732b0e2c1c7,virginia beach,7780,American Council of Engineering Companies,USA,2018-05-05 03:45:00,2018-05-07 03:52:00,0afc1c76dee6174b12ec4732b0e2c1c7,2018-05-05 03:45:00,NaT,VA,Transportation Engineer PE | Clark Nexsen,FALSE,https://jobopenings.acec.org/jobs/11010122/tra...,23452,2025-10-01 11:55:15.050,17-2051.01,"Clark Nexsen is a full-service architecture, e...",17-2051.01
3,0afd90aab2ab99306e51e2682d12c010,austin,56901,"Ferrovial, S.A.",USA,2024-08-23 08:28:00,2024-11-01 23:17:00,0afd90aab2ab99306e51e2682d12c010,2024-10-31 20:31:00,NaT,TX,Traffic and Revenue Engineer,FALSE,https://ferrovial.wd3.myworkdayjobs.com/Ferrov...,78719,2025-10-01 11:55:15.050,17-2051.01,Who is Cintra?\n \nCintra is the highways bus...,17-2051.01
4,0afe5c462c6c010dcd0738d17a0166bc,None,81769,Ikos,None,2023-09-09 05:10:00,2023-11-18 12:04:00,0afe5c462c6c010dcd0738d17a0166bc,2023-11-16 03:50:00,NaT,None,Ingenieur Zulassung (m/w/d),TRUE,https://www.ikosconsulting.com/index.php/en/in...,None,2025-10-01 11:55:15.050,17-2051.01,KOMPETENZ UND LEIDENSCHAFT f\u00fcr Technologi...,17-2051.01


## Step 8: Load Additional Tables (Remote & Full-Time)

Load remote work status and employment type information.

In [21]:
# Load remote tag
remote_tag = load_parquet_with_lowercase(
    test_path / 'remote-tag',
    table_name="remote tag"
)

if remote_tag is not None:
    # Filter to current records only (end_date IS NULL)
    remote_tag_current = remote_tag[remote_tag['end_date'].isna()]
    print(f"  Current remote tags: {len(remote_tag_current):,}")
    print(f"  Remote status distribution:")
    print(remote_tag_current['remote_detail'].value_counts())

print()

# Load full-time/part-time
full_time = load_parquet_with_lowercase(
    test_path / 'full-time-part-time',
    table_name="full-time/part-time"
)

if full_time is not None:
    # Filter to current records only
    full_time_current = full_time[full_time['end_date'].isna()]
    print(f"  Current employment type: {len(full_time_current):,}")
    print(f"  Employment type distribution:")
    print(full_time_current['fulltime_parttime'].value_counts())

Loading remote tag... ✓ Loaded 26,510 rows
  Current remote tags: 23,943
  Remote status distribution:
remote_detail
Hybrid    2476
Remote    1407
Name: count, dtype: int64

Loading full-time/part-time... ✓ Loaded 8,160 rows
  Current employment type: 8,129
  Employment type distribution:
fulltime_parttime
fulltime             6938
parttime              700
fulltime_parttime     491
Name: count, dtype: int64


## Step 9: Join Everything Together

In [26]:
if 'combined' in locals():
    # Join with remote tag (left join - keep all jobs)
    if remote_tag is not None:
        remote_tag_current = remote_tag[remote_tag['end_date'].isna()]
        combined = combined.merge(
            remote_tag_current[['job_hash', 'remote_detail']],
            on='job_hash',
            how='left'
        )
        print("Added remote work information")
    
    # Join with full-time info
    if full_time is not None:
        full_time_current = full_time[full_time['end_date'].isna()]
        combined = combined.merge(
            full_time_current[['job_hash', 'fulltime_parttime']],
            on='job_hash',
            how='left'
        )
        print("Added employment type information")
    
    print("\nComplete Dataset Summary:")
    print("=" * 60)
    print(f"Total rows: {len(combined):,}")
    print(f"Total columns: {len(combined.columns)}")
    
    if 'remote_detail' in combined.columns:
        print("\nRemote work distribution:")
        print(combined['remote_detail'].value_counts())
    
    if 'fulltime_parttime' in combined.columns:
        print("\nEmployment type distribution:")
        print(combined['fulltime_parttime'].value_counts())
    
    display(combined.head())

Added remote work information
Added employment type information

Complete Dataset Summary:
Total rows: 101,148
Total columns: 24


,base_hash,city,company_id,company_name,country,created,delete_date,job_hash,last_checked,last_updated,...,zip,_meta_loaded_at,onet_occupation_code,job_description,onet_occupation_code_desc,remote_detail_x,fulltime_parttime_x,category,remote_detail_y,fulltime_parttime_y
0,0aee91ddbdb3c5003dab6aed89ac249f,None,2850,Urban Outfitters,USA,2017-07-08 08:43:00,2017-07-10 12:14:00,0aee91ddbdb3c5003dab6aed89ac249f,2017-07-08 08:43:00,NaT,...,None,2025-10-01 11:55:15.050,19-3051.00,"Founded in 1970, Urban Outfitters (www.UrbanOu...",19-3051.00,None,NaN,urban_regional_planning,None,NaN
1,0aeef994dfe70cb8f7872a20041926d0,tempe,42834,WSP,USA,2022-04-23 00:19:00,2022-06-02 05:03:00,0aeef994dfe70cb8f7872a20041926d0,2022-05-31 17:30:00,NaT,...,85280,2025-10-01 11:55:15.050,19-3051.00,"Assistant Consultant, Transportation Planner\n...",19-3051.00,None,NaN,urban_regional_planning,None,NaN
2,0af10dd301f77747fc3e9f973143984e,Sacramento,975,State of California,USA,2015-01-17 11:42:00,2015-02-04 02:50:00,0af10dd301f77747fc3e9f973143984e,2015-02-01 19:12:00,NaT,...,94204,2025-10-01 11:55:15.050,19-3051.00,Please reference PARF 64-5-014 on your STD 678...,19-3051.00,None,NaN,urban_regional_planning,None,NaN
3,0af2c72a88a64aacce8aabed46eb8308,swindon,2153,Jacobs Engineering,GBR,2019-07-09 18:46:00,2019-10-17 04:52:00,0af2c72a88a64aacce8aabed46eb8308,2019-10-15 05:41:00,NaT,...,SN2,2025-10-01 11:55:15.050,19-3051.00,Jacobs leads the global professional services ...,19-3051.00,None,NaN,urban_regional_planning,None,NaN
4,0af2db74af5fd6b3aed2afef7648ab7a,burlington,9508,"City of Burlington, NC",USA,2022-01-25 12:55:00,2022-06-05 08:38:00,0af2db74af5fd6b3aed2afef7648ab7a,2022-06-03 08:32:00,NaT,...,27215,2025-10-01 11:55:15.050,19-3051.00,Job Description\n \nThis position in the City...,19-3051.00,None,NaN,urban_regional_planning,None,NaN


## Step 10: View Sample Job Description

In [28]:
if 'combined' in locals() and len(combined) > 0:
    # Show a complete job description
    sample = combined[combined['job_description'].notna()].iloc[0]
    
    print("=" * 60)
    print("SAMPLE JOB")
    print("=" * 60)
    print(f"Job Hash: {sample['job_hash']}")
    print(f"Title: {sample.get('title', 'N/A')}")
    print(f"Company: {sample.get('company_name', 'N/A')}")
    
    city = sample.get('city', 'N/A')
    state = sample.get('state', '')
    country = sample.get('country', 'N/A')
    location = f"{city}, {state}, {country}" if state else f"{city}, {country}"
    print(f"Location: {location}")
    
    print(f"Posted: {sample.get('created', 'N/A')}")
    
    if 'remote_detail' in sample.index:
        print(f"Remote: {sample['remote_detail']}")
    if 'fulltime_parttime' in sample.index:
        print(f"Employment: {sample['fulltime_parttime']}")
    if 'onet_occupation_code' in sample.index:
        print(f"ONET Code: {sample['onet_occupation_code']}")
    
    print("\n" + "-" * 60)
    print("Description:")
    print("-" * 60)
    desc = str(sample['job_description'])[:1500]  # First 1500 characters
    print(desc)
    if len(str(sample['job_description'])) > 1500:
        print("\n... (truncated)")
        print(f"\nFull length: {len(sample['job_description'])} characters")

SAMPLE JOB
Job Hash: 0aee91ddbdb3c5003dab6aed89ac249f
Title: Urban Outfitters: Temporary AutoCAD Assistant
Company: Urban Outfitters
Location: None, USA
Posted: 2017-07-08 08:43:00
ONET Code: 19-3051.00

------------------------------------------------------------
Description:
------------------------------------------------------------
Founded in 1970, Urban Outfitters (www.UrbanOutfitters.com) operates more than 200 stores in the United States, Canada, and Europe, all offering an eclectic mix of merchandise. We stock our stores with what we love, calling on our—and our customer's—interest in contemporary art, music, and fashion. From men's & women's apparel and accessories to items for the apartment, we offer a lifestyle-specific shopping experience for the educated, urban-minded individual in the 18 to 30 year-old range—both online and in our stores as well as through our catalog.

UO is seeking temporary help in our Planning and Allocation department!

We are currently looking for 

## Step 11: Process ALL 3 Categories

In [29]:
print("LOADING ALL 3 CATEGORIES")
print("=" * 60)

all_jobs = []

for i, category in enumerate(categories, 1):
    print(f"\n[{i}/{len(categories)}] Processing: {category}")
    print("-" * 60)
    
    category_path = BASE_PATH / category
    
    try:
        # Load job records
        job_records = load_parquet_with_lowercase(
            category_path / 'job-records',
            table_name="job records"
        )
        
        # Load job descriptions (takes time!)
        job_descriptions = load_parquet_with_lowercase(
            category_path / 'job-descriptions',
            table_name="job descriptions"
        )
        
        if job_records is None or job_descriptions is None:
            print(f"  ⚠️  Skipping {category} - failed to load data")
            continue
        
        # Join them
        combined = job_records.merge(
            job_descriptions,
            on='job_hash',
            how='inner',
            suffixes=('', '_desc')
        )
        print(f"  ✓ Joined: {len(combined):,} jobs with descriptions")
        
        # Load and join remote tag
        try:
            remote_tag = load_parquet_with_lowercase(
                category_path / 'remote-tag',
                table_name="remote tag"
            )
            if remote_tag is not None:
                remote_tag_current = remote_tag[remote_tag['end_date'].isna()]
                combined = combined.merge(
                    remote_tag_current[['job_hash', 'remote_detail']],
                    on='job_hash',
                    how='left'
                )
                print(f"  ✓ Added remote work info: {len(remote_tag_current):,} tags")
        except Exception as e:
            print(f"  ⚠️  Could not load remote tags: {e}")
        
        # Load and join full-time
        try:
            full_time = load_parquet_with_lowercase(
                category_path / 'full-time-part-time',
                table_name="employment type"
            )
            if full_time is not None:
                full_time_current = full_time[full_time['end_date'].isna()]
                combined = combined.merge(
                    full_time_current[['job_hash', 'fulltime_parttime']],
                    on='job_hash',
                    how='left'
                )
                print(f"  ✓ Added employment type: {len(full_time_current):,} tags")
        except Exception as e:
            print(f"  ⚠️  Could not load employment type: {e}")
        
        # Add category label
        combined['category'] = category.replace('extracted_', '')
        
        all_jobs.append(combined)
        print(f"  ✓ Category complete: {len(combined):,} rows")
        
    except Exception as e:
        print(f"  ❌ Error processing {category}: {e}")
        continue

# Combine all categories
if all_jobs:
    print("\n" + "=" * 60)
    print("COMBINING ALL CATEGORIES")
    print("=" * 60)
    
    full_dataset = pd.concat(all_jobs, ignore_index=True)
    
    print(f"\n✓ SUCCESS! Total jobs loaded: {len(full_dataset):,}")
    print("\nBreakdown by category:")
    print(full_dataset['category'].value_counts())
else:
    print("\n❌ No data loaded - check errors above")

LOADING ALL 3 CATEGORIES

[1/3] Processing: extracted_transportation_engineers
------------------------------------------------------------
Loading job records... ✓ Loaded 36,379 rows
Loading job descriptions... ✓ Loaded 24,111 rows
  ✓ Joined: 24,111 jobs with descriptions
Loading remote tag... ✓ Loaded 26,510 rows
  ✓ Added remote work info: 23,943 tags
Loading employment type... ✓ Loaded 8,160 rows
  ✓ Added employment type: 8,129 tags
  ✓ Category complete: 24,111 rows

[2/3] Processing: extracted_transportation_planners
------------------------------------------------------------
Loading job records... ✓ Loaded 5,748 rows
Loading job descriptions... ✓ Loaded 2,022 rows
  ✓ Joined: 2,022 jobs with descriptions
Loading remote tag... ✓ Loaded 2,129 rows
  ✓ Added remote work info: 2,015 tags
Loading employment type... ✓ Loaded 467 rows
  ✓ Added employment type: 465 tags
  ✓ Category complete: 2,022 rows

[3/3] Processing: extracted_urban_regional_planning
---------------------------

## Step 12: Dataset Statistics & Quality Check

In [30]:
if 'full_dataset' in locals():
    print("DATASET SUMMARY")
    print("=" * 60)
    
    print(f"\nTotal jobs: {len(full_dataset):,}")
    print(f"Total columns: {len(full_dataset.columns)}")
    print(f"Memory usage: {full_dataset.memory_usage(deep=True).sum() / 1_000_000:.1f} MB")
    
    print("\n" + "-" * 60)
    print("By Category:")
    print(full_dataset['category'].value_counts())
    
    print("\n" + "-" * 60)
    print("Date Range:")
    print(f"Oldest job: {full_dataset['created'].min()}")
    print(f"Newest job: {full_dataset['created'].max()}")
    years_span = (full_dataset['created'].max() - full_dataset['created'].min()).days / 365.25
    print(f"Span: {years_span:.1f} years")
    
    print("\n" + "-" * 60)
    print("Jobs with Descriptions:")
    with_desc = full_dataset['job_description'].notna().sum()
    print(f"{with_desc:,} ({with_desc/len(full_dataset)*100:.1f}%)")
    
    print("\n" + "-" * 60)
    print("Top 10 Companies:")
    if 'company_name' in full_dataset.columns:
        print(full_dataset['company_name'].value_counts().head(10))
    
    print("\n" + "-" * 60)
    print("Top 10 Cities:")
    if 'city' in full_dataset.columns:
        print(full_dataset['city'].value_counts().head(10))
    
    print("\n" + "-" * 60)
    print("Countries:")
    if 'country' in full_dataset.columns:
        print(full_dataset['country'].value_counts())
    
    print("\n" + "-" * 60)
    print("Data Quality Check:")
    print(f"Duplicate job_hashes: {full_dataset['job_hash'].duplicated().sum():,}")
    
    # Description length statistics
    desc_lengths = full_dataset['job_description'].str.len()
    print(f"\nDescription length:")
    print(f"  Average: {desc_lengths.mean():.0f} characters")
    print(f"  Median: {desc_lengths.median():.0f} characters")
    print(f"  Min: {desc_lengths.min():.0f} characters")
    print(f"  Max: {desc_lengths.max():.0f} characters")

DATASET SUMMARY

Total jobs: 127,281
Total columns: 22
Memory usage: 833.6 MB

------------------------------------------------------------
By Category:
category
urban_regional_planning     101148
transportation_engineers     24111
transportation_planners       2022
Name: count, dtype: int64

------------------------------------------------------------
Date Range:
Oldest job: 2012-01-08 07:55:00
Newest job: 2025-09-30 21:38:00
Span: 13.7 years

------------------------------------------------------------
Jobs with Descriptions:
127,281 (100.0%)

------------------------------------------------------------
Top 10 Companies:
company_name
AECOM                               6137
Aecom Technology Corporation        4237
State of California                 2366
Stantec Inc.                        2029
ARUP Group                          1917
WSP                                 1859
HNTB Corporation                    1835
Kimley-Horn and Associates, Inc.    1804
US Department of Transportat

## Step 13: Save Combined Dataset

In [31]:
if 'full_dataset' in locals():
    # Create output directory (go up one level from notebooks)
    output_dir = Path('../data/processed')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("SAVING DATASET")
    print("=" * 60)
    
    # Save as parquet (compressed, efficient for Python)
    output_parquet = output_dir / 'combined_dataset.parquet'
    full_dataset.to_parquet(output_parquet, compression='snappy')
    file_size_mb = output_parquet.stat().st_size / 1_000_000
    print(f"✓ Saved parquet: {output_parquet}")
    print(f"  File size: {file_size_mb:.1f} MB")
    
    # Save as CSV (can open in Excel)
    output_csv = output_dir / 'combined_dataset.csv'
    full_dataset.to_csv(output_csv, index=False)
    csv_size_mb = output_csv.stat().st_size / 1_000_000
    print(f"\n✓ Saved CSV: {output_csv}")
    print(f"  File size: {csv_size_mb:.1f} MB")
    
    print("\n" + "=" * 60)
    print("✓ DATA LOADING COMPLETE!")
    print("=" * 60)
    print(f"\nYour dataset is ready for analysis!")
    print(f"Total jobs: {len(full_dataset):,}")
    print(f"Saved to: {output_dir.absolute()}")
    print(f"\nFiles created:")
    print(f"  📄 combined_dataset.parquet ({file_size_mb:.1f} MB)")
    print(f"  📄 combined_dataset.csv ({csv_size_mb:.1f} MB)")
else:
    print("❌ No dataset to save - check errors above")

SAVING DATASET
✓ Saved parquet: ..\data\processed\combined_dataset.parquet
  File size: 353.8 MB

✓ Saved CSV: ..\data\processed\combined_dataset.csv
  File size: 685.5 MB

✓ DATA LOADING COMPLETE!

Your dataset is ready for analysis!
Total jobs: 127,281
Saved to: C:\Users\Nikhil\OneDrive\Documents\urban-ai-planning\notebooks\..\data\processed

Files created:
  📄 combined_dataset.parquet (353.8 MB)
  📄 combined_dataset.csv (685.5 MB)


## Step 14: Display Final Dataset Preview

In [32]:
if 'full_dataset' in locals():
    print("FINAL DATASET PREVIEW")
    print("=" * 60)
    print(f"Shape: {full_dataset.shape[0]:,} rows × {full_dataset.shape[1]} columns")
    print(f"\nColumns: {list(full_dataset.columns)}")
    print("\nFirst 10 rows:")
    display(full_dataset.head(10))

FINAL DATASET PREVIEW
Shape: 127,281 rows × 22 columns

Columns: ['base_hash', 'city', 'company_id', 'company_name', 'country', 'created', 'delete_date', 'job_hash', 'last_checked', 'last_updated', 'state', 'title', 'unmapped_location', 'url', 'zip', '_meta_loaded_at', 'onet_occupation_code', 'job_description', 'onet_occupation_code_desc', 'remote_detail', 'fulltime_parttime', 'category']

First 10 rows:


,base_hash,city,company_id,company_name,country,created,delete_date,job_hash,last_checked,last_updated,...,unmapped_location,url,zip,_meta_loaded_at,onet_occupation_code,job_description,onet_occupation_code_desc,remote_detail,fulltime_parttime,category
0,0afbae4c718319e9ca550c7d191257ce,Richland,32064,City of Richland,USA,2017-03-16 04:08:00,2017-06-21 21:28:00,0afbae4c718319e9ca550c7d191257ce,2017-06-19 20:40:00,NaT,...,FALSE,http://agency.governmentjobs.com/richlandwa/de...,99354,2025-10-01 11:55:15.050,17-2051.01,General Summary Benefits Supplemental Question...,17-2051.01,None,NaN,transportation_engineers
1,0afbfea6839cae2188306dc3de739c00,santa ana,975,State of California,USA,2023-07-13 08:34:00,2023-07-27 04:01:00,0afbfea6839cae2188306dc3de739c00,2023-07-24 10:51:00,NaT,...,FALSE,https://www.calcareers.ca.gov/CalHrPublic/Jobs...,92701,2025-10-01 11:55:15.050,17-2051.01,Job Description and Duties\n \nUnder the gene...,17-2051.01,None,fulltime,transportation_engineers
2,0afc1c76dee6174b12ec4732b0e2c1c7,virginia beach,7780,American Council of Engineering Companies,USA,2018-05-05 03:45:00,2018-05-07 03:52:00,0afc1c76dee6174b12ec4732b0e2c1c7,2018-05-05 03:45:00,NaT,...,FALSE,https://jobopenings.acec.org/jobs/11010122/tra...,23452,2025-10-01 11:55:15.050,17-2051.01,"Clark Nexsen is a full-service architecture, e...",17-2051.01,None,NaN,transportation_engineers
3,0afd90aab2ab99306e51e2682d12c010,austin,56901,"Ferrovial, S.A.",USA,2024-08-23 08:28:00,2024-11-01 23:17:00,0afd90aab2ab99306e51e2682d12c010,2024-10-31 20:31:00,NaT,...,FALSE,https://ferrovial.wd3.myworkdayjobs.com/Ferrov...,78719,2025-10-01 11:55:15.050,17-2051.01,Who is Cintra?\n \nCintra is the highways bus...,17-2051.01,None,fulltime,transportation_engineers
4,0afe5c462c6c010dcd0738d17a0166bc,None,81769,Ikos,None,2023-09-09 05:10:00,2023-11-18 12:04:00,0afe5c462c6c010dcd0738d17a0166bc,2023-11-16 03:50:00,NaT,...,TRUE,https://www.ikosconsulting.com/index.php/en/in...,None,2025-10-01 11:55:15.050,17-2051.01,KOMPETENZ UND LEIDENSCHAFT f\u00fcr Technologi...,17-2051.01,None,NaN,transportation_engineers
5,0b030653a4d631393552d3ed13fe77e2,raleigh,1934,State of North Carolina,USA,2025-03-06 03:12:00,2025-03-18 05:17:00,0b030653a4d631393552d3ed13fe77e2,2025-03-16 04:33:00,NaT,...,FALSE,https://www.governmentjobs.com/careers/northca...,27611,2025-10-01 11:55:15.050,17-2051.01,This position with NCDOT offers full State Ben...,17-2051.01,None,fulltime,transportation_engineers
6,0b0518311f6dad08e4ec272e4af07705,colonial heights,10384,State of Virginia,USA,2018-09-18 19:46:00,2018-09-21 14:46:00,0b0518311f6dad08e4ec272e4af07705,2018-09-18 19:46:00,NaT,...,FALSE,https://jobs.agencies.virginia.gov/applicants/...,None,2025-10-01 11:55:15.050,17-2051.01,Interested in building a career in traffic eng...,17-2051.01,None,NaN,transportation_engineers
7,0b0551a6a61f4776ec0cdab4c6746441,chicago,23391,T.Y. Lin International,USA,2020-07-02 05:20:00,2020-08-13 08:52:00,0b0551a6a61f4776ec0cdab4c6746441,2020-08-11 08:45:00,NaT,...,FALSE,https://careers-tylin.icims.com/jobs/2491/tran...,60602,2025-10-01 11:55:15.050,17-2051.01,Overview\n \nT.Y. Lin International is lookin...,17-2051.01,None,parttime,transportation_engineers
8,0b0887cfaff38bcdd83e5274d4f192d3,indianapolis,42834,WSP,USA,2024-01-03 21:45:00,2024-01-18 17:33:00,0b0887cfaff38bcdd83e5274d4f192d3,2024-01-17 19:15:00,NaT,...,FALSE,https://phe.tbe.taleo.net/phe01/ats/careers/v2...,46218,2025-10-01 11:55:15.050,17-2051.01,This Opportunity\n \nBe involved in exciting ...,17-2051.01,None,fulltime,transportation_engineers
9,bf8a9482f4c73a5a46b501983a0213eb,tysons corner,28425,Vanasse Hangen Brustlin,USA,2024-02-18 18:36:00,2024-06-11 18:15:00,1292e76cdd479097f2f717d800f1f414,2024-06-09 06:24:00,2024-04-26 04:29:00,...,FALSE,https://careers-vhb.icims.com/jobs/4267/transp...,None,2025-10-01 11:55:15.050,17-2051.01,Overview\n \nABOUT THE POSITION\n \nVHB is s...,17-2051.01,Hybrid,NaN,transportation_engineers


## I have successfully completed following steps : -

1. Loaded partitioned parquet files from all 3 categories
2. Handled UPPERCASE column names
3. Combined job records with descriptions
4. Added remote work and employment type information
5. Created one master dataset with 32,000+ jobs
6. Saved it for future analysis

In [33]:
print("ORIGINAL DATA (Before Joining)")
print("=" * 60)

# Count from each category separately
for category in categories:
    category_path = BASE_PATH / category
    
    # Load job records
    job_records = pd.read_parquet(category_path / 'job-records')
    job_records.columns = job_records.columns.str.lower()
    
    # Load job descriptions
    job_descriptions = pd.read_parquet(category_path / 'job-descriptions')
    job_descriptions.columns = job_descriptions.columns.str.lower()
    
    print(f"\n{category}:")
    print(f"  Total job records: {len(job_records):,}")
    print(f"  Job descriptions: {len(job_descriptions):,}")
    print(f"  Difference: {len(job_records) - len(job_descriptions):,} jobs WITHOUT descriptions")

ORIGINAL DATA (Before Joining)

extracted_transportation_engineers:
  Total job records: 36,379
  Job descriptions: 24,111
  Difference: 12,268 jobs WITHOUT descriptions

extracted_transportation_planners:
  Total job records: 5,748
  Job descriptions: 2,022
  Difference: 3,726 jobs WITHOUT descriptions

extracted_urban_regional_planning:
  Total job records: 131,820
  Job descriptions: 101,148
  Difference: 30,672 jobs WITHOUT descriptions


In [34]:
print("DATA INTEGRITY CHECK")
print("=" * 60)

# 1. Check for duplicates
duplicates = full_dataset['job_hash'].duplicated().sum()
print(f"\n1. Duplicate job_hashes: {duplicates:,}")
if duplicates == 0:
    print("   ✓ GOOD - Each job appears exactly once")
else:
    print("   ⚠️  WARNING - Some jobs duplicated!")

# 2. Check all jobs have descriptions
missing_desc = full_dataset['job_description'].isna().sum()
print(f"\n2. Jobs missing descriptions: {missing_desc:,}")
if missing_desc == 0:
    print("   ✓ GOOD - All jobs have descriptions")
else:
    print(f"   ⚠️  WARNING - {missing_desc} jobs have no description!")

# 3. Check all jobs have basic metadata
print("\n3. Missing key fields:")
key_fields = ['job_hash', 'title', 'company_name', 'created', 'category']
for field in key_fields:
    if field in full_dataset.columns:
        missing = full_dataset[field].isna().sum()
        pct = (missing / len(full_dataset)) * 100
        status = "✓" if missing == 0 else "⚠️"
        print(f"   {status} {field}: {missing:,} missing ({pct:.1f}%)")

# 4. Check optional fields (these CAN have missing values)
print("\n4. Optional fields (some missing is OK):")
optional_fields = ['remote_detail', 'fulltime_parttime', 'city', 'state']
for field in optional_fields:
    if field in full_dataset.columns:
        present = full_dataset[field].notna().sum()
        pct = (present / len(full_dataset)) * 100
        print(f"   {field}: {present:,} / {len(full_dataset):,} ({pct:.1f}%)")

# 5. Verify column count
print(f"\n5. Total columns: {len(full_dataset.columns)}")
print(f"   Columns: {list(full_dataset.columns)}")

print("\n" + "=" * 60)
print("INTEGRITY CHECK COMPLETE")
print("=" * 60)

DATA INTEGRITY CHECK

1. Duplicate job_hashes: 0
   ✓ GOOD - Each job appears exactly once

2. Jobs missing descriptions: 0
   ✓ GOOD - All jobs have descriptions

3. Missing key fields:
   ✓ job_hash: 0 missing (0.0%)
   ✓ title: 0 missing (0.0%)
   ✓ company_name: 0 missing (0.0%)
   ✓ created: 0 missing (0.0%)
   ✓ category: 0 missing (0.0%)

4. Optional fields (some missing is OK):
   remote_detail: 16,251 / 127,281 (12.8%)
   fulltime_parttime: 46,767 / 127,281 (36.7%)
   city: 113,900 / 127,281 (89.5%)
   state: 107,779 / 127,281 (84.7%)

5. Total columns: 22
   Columns: ['base_hash', 'city', 'company_id', 'company_name', 'country', 'created', 'delete_date', 'job_hash', 'last_checked', 'last_updated', 'state', 'title', 'unmapped_location', 'url', 'zip', '_meta_loaded_at', 'onet_occupation_code', 'job_description', 'onet_occupation_code_desc', 'remote_detail', 'fulltime_parttime', 'category']

INTEGRITY CHECK COMPLETE


In [35]:
print("FINAL DATASET VERIFICATION")
print("=" * 60)

# What you have
print(f"\nTotal jobs with descriptions: {len(full_dataset):,}")
print(f"Date range: {full_dataset['created'].min()} to {full_dataset['created'].max()}")
print(f"Categories: {full_dataset['category'].nunique()}")

# Sample a few jobs to verify they're complete
print("\n" + "-" * 60)
print("SAMPLE JOB (to verify completeness):")
print("-" * 60)

sample = full_dataset.sample(1).iloc[0]
print(f"Job Hash: {sample['job_hash']}")
print(f"Title: {sample.get('title', 'MISSING')}")
print(f"Company: {sample.get('company_name', 'MISSING')}")
print(f"Category: {sample.get('category', 'MISSING')}")
print(f"Created: {sample.get('created', 'MISSING')}")
print(f"Has description: {'YES' if pd.notna(sample.get('job_description')) else 'NO'}")
print(f"Description length: {len(str(sample['job_description']))} characters")
print(f"Remote info: {sample.get('remote_detail', 'Not available')}")
print(f"Employment type: {sample.get('fulltime_parttime', 'Not available')}")

print("\nFirst 200 characters of description:")
print(str(sample['job_description'])[:200] + "...")

FINAL DATASET VERIFICATION

Total jobs with descriptions: 127,281
Date range: 2012-01-08 07:55:00 to 2025-09-30 21:38:00
Categories: 3

------------------------------------------------------------
SAMPLE JOB (to verify completeness):
------------------------------------------------------------
Job Hash: 4d7be0bbcfa509fca246a90657f217a0
Title: Senior Transportation Planner- Part Time
Company: HDR, Inc.
Category: transportation_planners
Created: 2019-10-04 21:59:00
Has description: YES
Description length: 41 characters
Remote info: None
Employment type: parttime

First 200 characters of description:
About Us

At HDR, we specialize in engine...
